# PaliGemma Fine-tuning Dataset Preparation
This notebook converts our generated images and label text files into the **PaliGemma JSONL format**.

### 🧠 Entity Classes vs. Document Types
Even though we have 4 document types (**Passport**, **Drivers**, **SSN**, **Insurance**), our model's goal is to **detect the actual PII fields** *inside* those documents.

Our generated dataset contains 4 target entity classes:
1. `person_name`: Full legal names (found on all 4 types).
2. `ssn`: Social Security Numbers (found on SSN card).
3. `dob`: Dates of Birth (found on Drivers and Passport).
4. `national_id`: A general class for License #, Passport #, and Member ID.

In [ ]:
import os
import json
import glob
import random

# Paths
IMAGE_DIR = "dataset/images/"
LABEL_DIR = "dataset/labels/"
OUTPUT_JSONL = "dataset/metadata.jsonl"

# MATCHING THE 4 ENTITIES GENERATED IN OUR PREVIOUS NOTEBOOK
TARGET_CLASSES = "person_name; ssn; dob; national_id"
PROMPT_PREFIX = f"detect {TARGET_CLASSES}"

print(f"Scanning {IMAGE_DIR} for training data...")
images = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.png")))
print(f"Found {len(images)} images.")

dataset_entries = []

for img_path in images:
    img_filename = os.path.basename(img_path)
    label_filename = img_filename.replace(".png", ".txt")
    label_path = os.path.join(LABEL_DIR, label_filename)
    
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            ground_truth = f.read().strip()
            
        entry = {
            "image": img_filename,
            "prefix": PROMPT_PREFIX,
            "suffix": ground_truth
        }
        dataset_entries.append(entry)
    else:
        print(f"Warning: Label not found for {img_filename}")

print(f"\nSuccessfully formatted {len(dataset_entries)} examples.")

random.shuffle(dataset_entries)

with open(OUTPUT_JSONL, "w") as f:
    for entry in dataset_entries:
        f.write(json.dumps(entry) + "\n")

print(f"\nDataset saved to: {OUTPUT_JSONL}")

## Verify a Sample Entry

In [ ]:
if dataset_entries:
    print("Sample JSONL Entry:")
    print(json.dumps(dataset_entries[0], indent=2))
else:
    print("No entries found. Make sure to run the Synthetic Data Generation first!")